In [1]:
import os
import sys
import re

In [2]:
import numpy as np
import librosa
import pandas as pd
import sklearn
import scipy

In [3]:
import tensorflow as tf
import kagglehub

I0000 00:00:1779809039.905018   30422 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779809039.905372   30422 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779809039.938368   30422 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779809040.578089   30422 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

# Load the yamnet model

In [4]:
# Download latest version
yamnet_model_path = kagglehub.model_download("google/yamnet/tensorFlow2/yamnet")
print("Path to model files:", yamnet_model_path)

Path to model files: /home/nimcompoo/.cache/kagglehub/models/google/yamnet/tensorFlow2/yamnet/1


In [5]:
yamnet = tf.saved_model.load(yamnet_model_path)
yamnet_classes = pd.read_csv(str(yamnet.class_map_path().numpy(), encoding='utf-8'))

E0000 00:00:1779809041.563952   30422 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
yamnet_classes

,index,mid,display_name
0,0,/m/09x0r,Speech
1,1,/m/0ytgt,"Child speech, kid speaking"
2,2,/m/01h8n0,Conversation
3,3,/m/02qldy,"Narration, monologue"
4,4,/m/0261r1,Babbling
...,...,...,...
516,516,/m/07p_0gm,Throbbing
517,517,/m/01jwx6,Vibration
518,518,/m/07c52,Television
519,519,/m/06bz3,Radio


# Load the hornbase dataset

In [7]:
train = pd.read_csv('dataset/hornbase_train.csv')
test = pd.read_csv('dataset/hornbase_test.csv')

In [8]:
train['path'] = 'dataset/Dataset/' + train['file']
test['path'] = 'dataset/Dataset/' + test['file']

In [9]:
def get_embedding(path):
    audio, _ = librosa.load(path, sr=16000, mono=True)
    _, embed, _ = yamnet(audio)
    return embed

x_train = train['path'].apply(get_embedding).to_list()
x_test = test['path'].apply(get_embedding).to_list()

In [10]:
x_train = np.array(x_train)
x_test = np.array(x_test)

In [11]:
x_train[0].shape

(2, 1024)

# Train a neural network of the form,  2048 x 512 x 512 x 128 x 2

In [12]:
horn_detector = tf.keras.models.Sequential()

horn_detector.add( tf.keras.layers.Input(shape=(2, 1024)) )
horn_detector.add( tf.keras.layers.GlobalAveragePooling1D() )

horn_detector.add( tf.keras.layers.Dense(512, activation='relu') )
horn_detector.add( tf.keras.layers.BatchNormalization() )
horn_detector.add( tf.keras.layers.Dropout(0.3) )

horn_detector.add( tf.keras.layers.Dense(128, activation='relu') )
horn_detector.add( tf.keras.layers.BatchNormalization() )
horn_detector.add( tf.keras.layers.Dropout(0.2) )

horn_detector.add( tf.keras.layers.Dense(2, activation='softmax') )

In [13]:
horn_detector.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

In [14]:
horn_detector.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ global_average_pooling1d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 593,282 (2.26 MB)

 Trainable params: 592,002 (2.26 MB)

 Non-trainable params: 1,280 (5.00 KB)

In [15]:
y_train = train['class'] == 'horn'
y_test = test['class'] == 'horn'

In [16]:
horn_detector.fit(x_train, y_train, batch_size=128, epochs=32)

Epoch 1/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7526 - loss: 0.5939  
Epoch 2/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8757 - loss: 0.2988
Epoch 3/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9061 - loss: 0.2331 
Epoch 4/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9140 - loss: 0.2059 
Epoch 5/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9299 - loss: 0.1894 
Epoch 6/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9431 - loss: 0.1418 
Epoch 7/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9577 - loss: 0.1204 
Epoch 8/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9709 - loss: 0.0893 
Epoch 9/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9828 - loss: 0.0667 
Epoch 10/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9815 - loss: 0.0628 
Epoch 11/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9775 - loss: 0.0670 
Epoch 12/32
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9841 - loss: 0.0626 


In [17]:
(y_test == (horn_detector.predict(x_test)[:, 1] > 0.9)).mean()

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


np.float64(0.8549382716049383)

In [18]:
#horn_detector.save('horn-detector.keras')

In [19]:
#converter = tf.lite.TFLiteConverter.from_keras_model(horn_detector)
#horn_detector_lite = converter.convert()

In [20]:
#with open('horn_detector.tflite', 'wb') as k:
#    k.write(horn_detector_lite)